In [0]:
# =============================================================================
# NOTEBOOK  : silver_warehouse_inventory
# PURPOSE   : bronze.warehouse_inventory → silver.warehouse_inventory
# LAYER     : Silver (clean, validate, recompute, merge)
# FREQUENCY : Hourly (availableNow)
# TRIGGER   : availableNow
#
# TRANSFORM:
#   - last_updated     : String → TimestampType
#   - available_stock  : recomputed as current_stock - reserved_stock
#                        source derivation not trusted
#   - stock columns    : already BIGINT from source, keep as-is
#
# MERGE & DEDUP LOGIC:
#   Merge key : warehouse_id + product_id
#               latest snapshot per warehouse-product pair
#   Dedup     : dropDuplicates on (warehouse_id, product_id) before merge
#               hourly snapshots may have duplicate pairs in same microbatch
#               keep the row with latest last_updated within each pair
#
#   Case 1 : warehouse_id + product_id NOT in silver → INSERT
#   Case 2 : warehouse_id + product_id IN silver, stock changed → UPDATE
#            all stock columns are updateable, except location zone — this is a snapshot table
#            every new snapshot is potentially an update
#   Case 3 : warehouse_id + product_id IN silver, no change → IGNORE
#
# NOTE: This is a SNAPSHOT table — silver always reflects CURRENT state
#       not historical. Every hourly snapshot overwrites previous values.
#       Historical stock levels are tracked via Delta time travel if needed.
# =============================================================================

# ── Imports & Config ──────────────────────────────────────────────────
import sys, json
sys.path.insert(0, "/Workspace/Shared/mini_project_a3t7")

from utils.audit import start_audit, end_audit
from utils.schema_registry import SILVER_WAREHOUSE_INVENTORY_SCHEMA

from pyspark.sql.functions import (
    current_timestamp, lit, col,
    to_timestamp, expr
)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from delta.tables import DeltaTable

with open("/Workspace/Shared/mini_project_a3t7/config/config.json") as f:
    cfg = json.load(f)

SOURCE_TABLE = cfg["tables"]["bronze_warehouse_inventory"]
TARGET_TABLE = cfg["tables"]["silver_warehouse_inventory"]
CHECKPOINT   = cfg["checkpoint_paths"]["silver_warehouse_inventory"]
PIPELINE     = "silver_warehouse_inventory"

In [0]:


# ── foreachBatch function + Stream ────────────────────────────────────

def process_microbatch(micro_batch_df, microbatch_id):
    """
    Called by Spark for every micro-batch of bronze warehouse inventory rows.

    DEDUP  : dropDuplicates not sufficient here — we need the LATEST snapshot
             per warehouse_id + product_id pair, not just any one
             Use window function ordered by last_updated DESC to keep latest

    MERGE  :
      Case 1 → warehouse_id + product_id not in silver → INSERT
      Case 2 → pair in silver, any stock value changed  → UPDATE all stock cols
      Case 3 → pair in silver, no change                → IGNORE

    NOTE   : available_stock is RECOMPUTED here
             source value not trusted — derived as current_stock - reserved_stock
    """

    import sys
    sys.path.insert(0, "/Workspace/Shared/mini_project_a3t7")
    from utils.audit import start_audit, end_audit
    from utils.schema_registry import SILVER_WAREHOUSE_INVENTORY_SCHEMA

    if micro_batch_df.isEmpty():
        return

    run_id = start_audit(spark, PIPELINE, TARGET_TABLE, SOURCE_TABLE)

    try:
        rows_read = micro_batch_df.count()

        # ── TRANSFORM ─────────────────────────────────────────────────────────
        df = (
            micro_batch_df

            # 1. last_updated: String → TimestampType
            #    Source format: "2022-01-01 00:00:00" (string from Parquet BINARY)
            .withColumn(
                "last_updated",
                to_timestamp(col("last_updated"))
            )

            # 2. Recompute available_stock — never trust source derivation
            #    available_stock = current_stock - reserved_stock
            #    source may have stale or incorrect derived values
            .withColumn(
                "available_stock",
                col("current_stock") - col("reserved_stock")
            )

            # 3. Silver audit columns
            .withColumn("processed_at",    current_timestamp())
            .withColumn("pipeline_run_id", lit(run_id))
            .withColumn("source_system",   lit("warehouse_inventory_parquet"))

            # 4. Enforce silver schema
            .select([f.name for f in SILVER_WAREHOUSE_INVENTORY_SCHEMA.fields])
        )

        # ── DEDUP WITHIN MICRO-BATCH ──────────────────────────────────────────
        # Simple dropDuplicates not enough here
        # If same warehouse_id + product_id appears twice in microbatch
        # we want the LATEST snapshot (highest last_updated), not just any one
        # Window ordered by last_updated DESC — row 1 = most recent snapshot
        window = (
            Window
            .partitionBy("warehouse_id", "product_id")
            .orderBy(col("last_updated").desc())
        )

        df = (
            df
            .withColumn("_row_num", row_number().over(window))
            .filter(col("_row_num") == 1)
            .drop("_row_num")
        )

        rows_after_dedup = df.count()
        if rows_read != rows_after_dedup:
            print(f"[DEDUP] kept latest snapshot, "
                  f"dropped {rows_read - rows_after_dedup} older snapshots "
                  f"in microbatch_id={microbatch_id}")

        # ── MERGE INTO SILVER ──────────────────────────────────────────────────
        # Merge key: warehouse_id + product_id (composite key)
        # Case 2: any stock value changed → UPDATE all stock columns + metadata
        # Case 1: new pair → INSERT
        # Case 3: no change → no rule fires → IGNORE
        (
            DeltaTable.forName(spark, TARGET_TABLE).alias("t")
            .merge(
                df.alias("s"),
                """
                t.warehouse_id = s.warehouse_id AND
                t.product_id   = s.product_id
                """
            )
            .whenMatchedUpdate(
                condition="""
                    t.current_stock   != s.current_stock   OR
                    t.reserved_stock  != s.reserved_stock  OR
                    t.available_stock != s.available_stock OR
                    t.reorder_level   != s.reorder_level   OR
                    t.max_stock       != s.max_stock
                """,
                set={
                    "current_stock":   "s.current_stock",
                    "reserved_stock":  "s.reserved_stock",
                    "available_stock": "s.available_stock",
                    "reorder_level":   "s.reorder_level",
                    "max_stock":       "s.max_stock",
                    "last_updated":    "s.last_updated",
                    "processed_at":    "current_timestamp()",
                    "pipeline_run_id": f"'{run_id}'"
                }
            )
            .whenNotMatchedInsertAll()
            .execute()
        )

        metrics = (
            spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE} LIMIT 1")
            .select("operationMetrics")
            .collect()[0][0]
        )
        rows_inserted = int(metrics.get("numTargetRowsInserted", 0))
        rows_updated  = int(metrics.get("numTargetRowsUpdated", 0))

        end_audit(
            spark, run_id, "SUCCESS",
            rows_read=rows_read,
            rows_written=rows_inserted + rows_updated,
            extra={
                "rows_inserted": str(rows_inserted),
                "rows_updated":  str(rows_updated),
                "microbatch_id": str(microbatch_id)
            }
        )

    except Exception as e:
        end_audit(spark, run_id, "FAILED", error=str(e))
        raise


# ── READ BRONZE AS STREAM ─────────────────────────────────────────────────────
bronze_stream = (
    spark.readStream
    .format("delta")
    .option("ignoreChanges", "true")
    .table(SOURCE_TABLE)
)

# ── WRITE WITH foreachBatch ───────────────────────────────────────────────────
query = (
    bronze_stream.writeStream
    .foreachBatch(process_microbatch)
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

In [0]:
# ── CELL 3: Verify last run ───────────────────────────────────────────────────

# 1. Audit status
spark.read.table("azure3_team7_project.platform.pipeline_audit") \
    .filter(col("pipeline_name") == PIPELINE) \
    .orderBy(col("start_time").desc()) \
    .limit(1) \
    .select("status", "rows_read", "rows_written", "extra_metadata") \
    .show(truncate=False)

# 2. Silver row count
print("Silver row count:", spark.read.table(TARGET_TABLE).count())

# 3. Nulls in key columns
print("Null key columns:",
    spark.read.table(TARGET_TABLE)
    .filter(col("warehouse_id").isNull() | col("product_id").isNull())
    .count()
)

# 4. Delta history
spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE} LIMIT 1") \
    .select("operation", "operationMetrics") \
    .show(truncate=False)